# Recreación de Wang, Xu & Zhou (2015) con datos simulados
**Large Unbalanced Credit Scoring Using Lasso-Logistic Regression Ensemble** · PLoS ONE 10(2): e0117844

Este es el notebook principal de la Actividad Final de Machine Learning II. Presenta el análisis, los resultados y su interpretación. El código reutilizable está en `src/` y los notebooks de detalle (`01_simulacion_datos`, `02_analisis_articulo`, `03_extension`) contienen la ejecución paso a paso; este notebook lee los resultados que esos notebooks guardan en `results/`.

| Sección | Contenido |
|---|---|
| 1 | Artículo y objetivo del estudio |
| 2 | Datos simulados: qué se reproduce y qué se supone |
| 3 | Recreación del análisis (Paso 5) |
| 4 | Comparación con el artículo (Paso 6) |
| 5 | Extensión experimental: discretización en rangos (Paso 7) |
| 6 | Conclusiones (Paso 8) |

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

warnings.filterwarnings("ignore")
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RES, FIG = RAIZ / "results", RAIZ / "figures"
ORDEN = ["LLRE", "RF", "LLR", "CART"]
pd.options.display.float_format = "{:.4f}".format

## 1. Artículo y objetivo del estudio
Wang, Xu & Zhou proponen el **LLRE** (*Lasso-Logistic Regression Ensemble*) para *credit scoring* con datos grandes y muy desbalanceados (*Give Me Some Credit*: 150.000 clientes, 6,7 % de impagos, razón ≈ 14:1).

**Método.** (1) Se agrupa la clase mayoritaria con *K-means* en *k* = 14 subgrupos. (2) Cada subgrupo se combina con una muestra *bootstrap* de la clase minoritaria, lo que da un conjunto balanceado. (3) Sobre cada conjunto se ajusta una regresión logística con penalización Lasso; el parámetro λ se elige con el AUC calculado sobre los datos que quedaron fuera del conjunto balanceado. (4) Las probabilidades de los modelos se promedian con pesos `w = 1 / (1 + e^(-AUC))`.

**Evaluación del artículo.** Se comparan LLRE, Random Forest (500 árboles, `mtry` = 3), Lasso-logística (LLR) y CART, con AUC y F-measure, sobre 30 particiones aleatorias 80 % / 20 % y dos conjuntos de variables: las 10 originales y 80 derivadas (logaritmos, banderas de valores especiales y faltantes, *binning*).

**Resultados del artículo.** El LLRE obtiene el mayor AUC con variables derivadas y el mayor F-measure; el Lasso-logístico pasa de un AUC de 0,4898 con variables originales a 0,8567 con derivadas; CART tiene F-measure igual a 0 en las 30 corridas.

## 2. Datos simulados
El artículo no publica los datos, solo su descripción. La simulación (`src/simulate_data.py`, semilla 42) reproduce exactamente lo que el artículo reporta y declara como supuesto todo lo demás. El detalle está en `notebooks/01_simulacion_datos.ipynb` y en la sección 3 del README.

In [ ]:
verif = pd.read_csv(RES / "paso4_verificacion.csv")
display(verif)
faltantes = pd.read_csv(RES / "paso4_faltantes.csv", index_col=0)
display(faltantes[faltantes["% faltantes"] > 0])

**Reportado por el artículo (se reproduce exacto):** tamaño de la muestra, conteo de cada clase, número de variables, y mínimo, máximo y media de `age`.

**Supuesto (no reportado):** forma de la distribución de cada variable, desviación estándar de `age`, porcentaje de faltantes (≈ 20 % en `MonthlyIncome`, ≈ 2,6 % en `NumberOfDependents`), códigos especiales 96/98 en las variables de atraso (≈ 0,18 % de las filas), independencia entre variables dado la clase y, sobre todo, la **intensidad de la relación de cada variable con el impago**. Esta última determina qué tan separables son las clases y, con ello, el nivel absoluto de AUC y F-measure.

Los valores del artículo no se usaron para calibrar la simulación.

In [ ]:
display(Image(filename=str(FIG / "paso4_distribuciones_por_clase.png")))

## 3. Recreación del análisis (Paso 5)
Se implementaron LLRE, RF, LLR y CART (`src/llre.py`, `src/evaluate.py`) y se evaluaron con el protocolo del artículo: partición aleatoria 80 % / 20 % por corrida, imputación, escalado y *clustering* ajustados solo con entrenamiento, AUC y F-measure de la clase positiva (umbral 0,5) sobre el conjunto de prueba, con variables originales y derivadas. Las variables derivadas son una reconstrucción hecha con las técnicas que el artículo declara, porque no lista sus 80 variables (`src/features.py`).

Las corridas, su orden y las semillas están en `results/paso5_resultados_por_corrida.csv`.

In [ ]:
res = pd.read_csv(RES / "paso5_resultados_por_corrida.csv")
N_RUNS = res["run"].nunique()
print(f"Corridas por combinación clasificador x variables: {N_RUNS} (el artículo usa 30)")

resumen = (res.groupby(["clasificador", "variables"])[["auc", "f1"]].agg(["mean", "std"]))
resumen.columns = [f"{m}_{s}" for m, s in resumen.columns]
resumen = resumen.reindex(pd.MultiIndex.from_product([ORDEN, ["original", "derived"]],
                                                     names=["clasificador", "variables"]))
resumen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, met, tit in zip(axes, ["auc", "f1"], ["AUC", "F-measure"]):
    for j, (modo, col, etq) in enumerate([("original", "#8e9aaf", "originales"), ("derived", "#3b6ea5", "derivadas")]):
        datos = [res[(res.clasificador == c) & (res.variables == modo)][met] for c in ORDEN]
        pos = np.arange(len(ORDEN)) + (j - 0.5) * 0.35
        bp = ax.boxplot(datos, positions=pos, widths=0.3, patch_artist=True)
        for b in bp["boxes"]:
            b.set_facecolor(col)
        ax.plot([], [], color=col, lw=8, label=etq)
    ax.set_xticks(range(len(ORDEN))); ax.set_xticklabels(ORDEN)
    ax.set_title(tit); ax.legend(); ax.grid(alpha=0.25)
fig.suptitle(f"Resultados simulados por clasificador ({N_RUNS} corridas)")
fig.tight_layout()
plt.show()

## 4. Comparación con el artículo (Paso 6)
Las tablas del artículo se transcribieron a `results/articulo/` (Tabla 3: AUC medio por conjunto de variables; Tablas 4 y 5: AUC y F-measure de cada una de las 30 corridas). Los valores absolutos dependen de la intensidad de la señal, que en la simulación es un supuesto, por lo que la comparación se centra en los patrones.

### 4.1 AUC

In [ ]:
t3 = pd.read_csv(RES / "articulo" / "tabla3_auc_por_conjunto_de_variables.csv").set_index("clasificador")
t4 = pd.read_csv(RES / "articulo" / "tabla4_auc_30_corridas.csv")
t5 = pd.read_csv(RES / "articulo" / "tabla5_fmeasure_30_corridas.csv")

auc_sim = res.pivot_table(index="clasificador", columns="variables", values="auc").loc[ORDEN]
f1_sim = res.pivot_table(index="clasificador", columns="variables", values="f1").loc[ORDEN]

comp_auc = pd.DataFrame({
    "artículo, originales (T3)": t3.loc[ORDEN, "AUC_originales"],
    "simulado, originales": auc_sim["original"],
    "artículo, derivadas (T3)": t3.loc[ORDEN, "AUC_derivadas"],
    "artículo, derivadas, media 30 corridas (T4)": t4[ORDEN].mean(),
    "simulado, derivadas": auc_sim["derived"],
})
comp_auc.to_csv(RES / "paso6_comparacion_auc.csv")
comp_auc

### 4.2 F-measure

In [ ]:
comp_f1 = pd.DataFrame({
    "artículo, media 30 corridas (T5)": t5[ORDEN].mean(),
    "artículo, desv. est.": t5[ORDEN].std(),
    "simulado, originales": f1_sim["original"],
    "simulado, derivadas": f1_sim["derived"],
})
comp_f1.to_csv(RES / "paso6_comparacion_f1.csv")
comp_f1

La Tabla 5 del artículo no indica el conjunto de variables; se asume el de variables derivadas, porque las medias de AUC de la Tabla 4 coinciden con las de la columna de derivadas de la Tabla 3.

### 4.3 Patrones cualitativos
Cada fila corresponde a una afirmación del artículo. El valor del artículo se calcula con sus tablas y el simulado con los resultados de la sección 3.

In [ ]:
def patrones():
    A = lambda c, v: auc_sim.loc[c, v]
    art_auc = lambda c, v: t3.loc[c, "AUC_originales" if v == "original" else "AUC_derivadas"]
    art_f1 = t5[ORDEN].mean()
    sim_f1 = f1_sim["derived"]
    filas = [
        ("LLR mejora al pasar de variables originales a derivadas (diferencia de AUC)",
         art_auc("LLR", "derived") - art_auc("LLR", "original"), A("LLR", "derived") - A("LLR", "original"),
         lambda x: x > 0.05),
        ("RF casi no cambia entre conjuntos de variables (|diferencia de AUC|)",
         abs(art_auc("RF", "derived") - art_auc("RF", "original")), abs(A("RF", "derived") - A("RF", "original")),
         lambda x: x < 0.02),
        ("LLRE supera a LLR con variables originales (diferencia de AUC)",
         art_auc("LLRE", "original") - art_auc("LLR", "original"), A("LLRE", "original") - A("LLR", "original"),
         lambda x: x > 0),
        ("CART tiene el menor AUC con variables derivadas (AUC de CART menos el mejor otro)",
         art_auc("CART", "derived") - max(art_auc(c, "derived") for c in ORDEN[:3]),
         A("CART", "derived") - max(A(c, "derived") for c in ORDEN[:3]),
         lambda x: x < 0),
        ("CART no predice la clase minoritaria (F-measure de CART)",
         art_f1["CART"], sim_f1["CART"], lambda x: x < 0.05),
        ("LLRE tiene el mayor F-measure (F-measure de LLRE menos el mejor otro)",
         art_f1["LLRE"] - art_f1[["RF", "LLR", "CART"]].max(), sim_f1["LLRE"] - sim_f1[["RF", "LLR", "CART"]].max(),
         lambda x: x > 0),
        ("LLRE tiene el mayor AUC con variables derivadas (AUC de LLRE menos el mejor otro)",
         art_auc("LLRE", "derived") - max(art_auc(c, "derived") for c in ORDEN[1:]),
         A("LLRE", "derived") - max(A(c, "derived") for c in ORDEN[1:]),
         lambda x: x > 0),
    ]
    out = pd.DataFrame([{"patrón": t, "artículo": a, "simulado": s,
                         "artículo cumple": f(a), "simulado cumple": f(s)} for t, a, s, f in filas])
    out["se reproduce"] = out["artículo cumple"] == out["simulado cumple"]
    return out

pat = patrones()
pat.to_csv(RES / "paso6_patrones.csv", index=False)
pd.set_option("display.max_colwidth", 100)
pat

### 4.4 Umbral de decisión del LLRE
La sección 5.7 del notebook `02_analisis_articulo` analiza por qué el LLRE puede tener un AUC comparable al de los demás modelos y, aun así, un F-measure menor. Se compara el umbral de 0,5 del artículo con una corrección por el cambio de prevalencia y con el mejor umbral posible sobre prueba (cota superior, no utilizable en la práctica). Este análisis no forma parte del artículo.

In [ ]:
pd.read_csv(RES / "paso5_umbral_llre.csv")

### 4.5 Interpretación

**Nivel absoluto de las métricas.** Con variables derivadas, el AUC simulado está entre 0,89 y 0,93 y el del artículo entre 0,77 y 0,86. La simulación produce clases más separables que los datos reales. La causa es que la intensidad de la señal es un supuesto (sección 2): las variables se generan independientes entre sí dado la clase y cada una está desplazada según la clase, mientras que en datos reales hay ruido y dependencias entre variables que reducen la separabilidad. La simulación no se calibró para igualar el AUC del artículo. Un mayor nivel de separabilidad cambia sobre todo las métricas que dependen del umbral (F-measure), como se ve más abajo.

**Patrones que se reproducen (4 de 7).** Los cuatro son de AUC.
- *LLR mejora con variables derivadas.* El AUC pasa de 0,734 a 0,935 (artículo: 0,490 a 0,857). La dirección coincide y la magnitud es menor.
- *RF casi no depende del conjunto de variables.* La diferencia es de 0,0008 (artículo: 0,010).
- *LLRE supera a LLR con variables originales.* La diferencia es de +0,184 (artículo: +0,190). Sin embargo, el LLRE simulado casi no cambia entre conjuntos de variables (0,918 con originales y 0,918 con derivadas), mientras que en el artículo mejora de 0,680 a 0,860. Es decir, la ventaja sobre LLR se reproduce, pero no por el mismo mecanismo.
- *CART tiene el menor AUC con variables derivadas.* Queda 0,040 por debajo del mejor modelo (artículo: 0,097). Además, CART mejora con variables derivadas en la simulación (0,857 a 0,895), mientras que en el artículo no cambia (0,770 a 0,763).

Una hipótesis para el bajo AUC de LLR con variables originales (0,734) es la presencia de los códigos especiales 96 y 98 en las variables de atraso: son valores extremos que dominan la varianza de esas columnas y no guardan una relación monótona con el impago, lo que perjudica a un modelo lineal. Los modelos de árboles no se ven afectados por valores extremos y las variables derivadas incluyen una bandera para esos códigos. Esta hipótesis es consistente con los resultados, pero no se verificó con un experimento específico.

**Patrones que no se reproducen (3 de 7).**
1. *CART no predice la clase minoritaria.* En el artículo el F-measure de CART es 0 en las 30 corridas; en la simulación es 0,653. Un árbol clasifica como moroso solo si la proporción de morosos en la hoja supera 0,5. Con la señal débil de los datos reales ninguna hoja lo logra, y con la señal más fuerte de la simulación varias hojas lo logran. Esta explicación es una hipótesis: no se probó variando la intensidad de la señal.
2. *LLRE tiene el mayor F-measure.* En el artículo el LLRE obtiene 0,384 frente a 0,275 de RF. En la simulación el LLRE obtiene 0,333, un valor cercano al del artículo, pero los demás clasificadores obtienen alrededor de 0,65, más del doble de los valores del artículo. La sección 4.4 muestra que el problema del LLRE es el umbral y no el ordenamiento de los clientes. Con el umbral de 0,5 marca 8.137 de 30.000 clientes como morosos (recall 0,894, precisión 0,212, F1 0,343), porque se entrena con conjuntos balanceados y sus probabilidades sobreestiman el riesgo respecto de la prevalencia real (6,7 %). Con un umbral de 0,83 el F1 sube a 0,648, similar al de RF (0,644) y LLR (0,646). Este último umbral se eligió con el conjunto de prueba y es una cota superior, no una regla utilizable; la corrección por prevalencia (b) sobrecorrige: precisión 0,968 pero recall 0,295. En el régimen del artículo (señal débil), RF con umbral 0,5 detecta pocos morosos y su F-measure es bajo, de modo que el sesgo del LLRE hacia marcar más clientes le favorece. En el régimen de la simulación, RF, LLR y CART ya alcanzan un buen equilibrio entre precisión y recall con 0,5, y el sesgo del LLRE lo perjudica.
3. *LLRE tiene el mayor AUC.* En el artículo la ventaja del LLRE es de 0,001 sobre el mejor competidor, menor que la desviación estándar entre corridas (≈ 0,004). En la simulación el LLRE queda por debajo de LLR (−0,017) y de RF (−0,008). Dos explicaciones posibles, sin verificar: cada modelo base del LLRE ve solo una fracción de la clase mayoritaria (un subgrupo de los 14), mientras que LLR usa todos los datos; y como los datos simulados siguen una estructura casi log-lineal con variables independientes dado la clase, un único modelo logístico bien especificado es difícil de superar.

**Resumen de la comparación.** La simulación reproduce la sensibilidad de los modelos lineales a la construcción de variables y la insensibilidad de RF. No reproduce la superioridad del LLRE ni el comportamiento de CART y de RF en F-measure. Con estos datos simulados, el LLRE no supera a RF ni a LLR en AUC y, con el umbral de 0,5, queda por debajo en F-measure.

### 4.6 Limitaciones
- **Número de corridas.** Se usaron 3 corridas y el artículo usa 30. Con 3 corridas no es posible aplicar la prueba de Friedman ni el post-hoc de Nemenyi del artículo; las diferencias de AUC menores a 0,01 no se pueden interpretar.
- **Datos simulados.** La intensidad de la señal, la forma de las distribuciones y la independencia entre variables son supuestos. Las conclusiones son sobre estos datos simulados y no sobre el dataset real.
- **Variables derivadas.** El artículo no lista sus 80 variables; el conjunto usado tiene 56 columnas y es una reconstrucción con las técnicas declaradas.
- **Implementación.** El artículo usa R (`glmnet`, `rpart`, `randomForest`); aquí se usó scikit-learn. La rejilla de λ tiene 10 valores (glmnet usa 100), la validación cruzada de LLR usa 3 pliegues (glmnet usa 10) y la tolerancia de los optimizadores es 1e-3. En LLR con variables derivadas el λ elegido queda cerca del extremo menos penalizado de la rejilla (columna `pos_rejilla`, notebook 02), por lo que su AUC podría estar levemente subestimado.
- **Hipótesis.** Las explicaciones marcadas como hipótesis en la sección 4.5 no se probaron con experimentos específicos.

## 5. Extensión experimental: discretización en rangos (Paso 7)
**Condición que se modifica:** la forma en que el modelo recibe las variables. Las 10 variables se reemplazan por indicadores de rango (*k* rangos por variable, con *k* = 2, 3, 5 y 10; los cortes salen de los cuantiles del conjunto de entrenamiento y las variables con valores faltantes tienen un rango adicional "NA"). Los datos simulados, las particiones 80 % / 20 % (mismas semillas del Paso 5), los cuatro clasificadores y sus hiperparámetros no cambian. El código está en `notebooks/03_extension.ipynb`.

**Motivación.** El trabajo de grado estudia un esquema de *scoring* en el que el modelo recibe rangos atestados mediante pruebas de conocimiento cero, en lugar de valores exactos. La pregunta es cuánto desempeño se pierde cuando la información de entrada se reduce a rangos.

**Hipótesis formuladas antes de ver los resultados.**
1. El desempeño disminuye al reducir *k*; con *k* = 2 la caída es la mayor.
2. Con *k* moderado (5 a 10) el desempeño se mantiene cerca del obtenido con variables derivadas.

In [ ]:
ext = pd.read_csv(RES / "paso7_resumen.csv")
KS = [2, 3, 5, 10]
ORDEN_VARS = ["original", "derived"] + [f"discretized_k{k}" for k in KS]
print(f"Corridas por combinación: {N_RUNS}")
print("AUC medio")
auc7 = ext.pivot(index="clasificador", columns="variables", values="auc_mean").loc[ORDEN, ORDEN_VARS]
display(auc7)
print("F-measure medio")
f17 = ext.pivot(index="clasificador", columns="variables", values="f1_mean").loc[ORDEN, ORDEN_VARS]
display(f17)

### 5.1 Variación respecto de las variables derivadas

In [ ]:
var7 = pd.read_csv(RES / "paso7_variacion_vs_derivadas.csv")
print("Diferencia de AUC (discretizadas menos derivadas)")
display(var7.pivot(index="clasificador", columns="k", values="dif_AUC_vs_derived").loc[ORDEN])
print("Diferencia de F-measure (discretizadas menos derivadas)")
display(var7.pivot(index="clasificador", columns="k", values="dif_F1_vs_derived").loc[ORDEN])
display(Image(filename=str(FIG / "paso7_discretizacion.png")))

### 5.2 Interpretación

**Hipótesis 1 (el desempeño cae al reducir *k*).** Se cumple solo para LLR, cuyo AUC es 0,934, 0,932, 0,927 y 0,914 para *k* = 10, 5, 3 y 2 (la caída con *k* = 2 respecto de las variables derivadas es de 0,021). No se cumple para los otros tres clasificadores:
- *RF* cae más con *k* = 3 (−0,041) que con *k* = 2 (−0,028).
- *CART* empeora al aumentar *k*: −0,008 con *k* = 2 y −0,042 con *k* = 10.
- *LLRE* varía menos de 0,007 en AUC entre todos los valores de *k*. Entre corridas, el AUC del LLRE para un mismo *k* llega a variar cerca de 0,015, de modo que esas diferencias no se distinguen de cero con 3 corridas.

**Hipótesis 2 (con *k* de 5 a 10 el desempeño se mantiene cerca del de las variables derivadas).** En AUC se cumple para LLR (−0,003 con *k* = 5 y −0,001 con *k* = 10) y para LLRE (−0,004 y −0,001); con *k* = 10 también para RF (−0,008). No se cumple para CART (−0,041 y −0,042) ni para RF con *k* = 5 (−0,020). En F-measure solo se cumple para LLR (−0,004 y −0,001); RF baja entre 0,045 y 0,066, CART entre 0,037 y 0,045, y el F-measure del LLRE cae a 0,14.

**Lectura por clasificador** (las explicaciones son hipótesis; no se probaron con experimentos específicos).
- *LLR* es el único que se comporta como se esperaba: con *k* ≥ 5 el AUC y el F-measure son prácticamente los de las variables derivadas, y con *k* = 2 la pérdida es de 0,02 en AUC y de 0,09 en F-measure. Además, con rangos el AUC (0,914 a 0,934) es mucho mayor que con las variables originales (0,734). Esto es consistente con la hipótesis de la sección 4.5: los rangos agrupan los valores extremos (códigos 96 y 98) con los demás valores altos, sin necesidad de una bandera explícita.
- *RF* usa `mtry` = 3 como en el artículo. Al expandir cada variable en indicadores, el número de columnas crece con *k* y cada división evalúa solo 3 de ellas, muchas de variables con poca señal. El hiperparámetro no se reajustó para este nuevo espacio de entrada.
- *CART* usa un criterio de parada equivalente al `cp` = 0,01 de `rpart` (la división debe reducir la impureza en al menos 1 % de la de la raíz). Un indicador de un rango estrecho separa pocos datos y rara vez cumple ese mínimo, por lo que el árbol se detiene antes cuando *k* es mayor.
- *LLRE* mantiene el AUC casi constante, pero su F-measure con umbral 0,5 baja de 0,329 (*k* = 2) a 0,140 (*k* = 10). Es el efecto de umbral descrito en la sección 4.4: el F-measure con 0,5 refleja la calibración de sus probabilidades más que su capacidad de ordenar clientes. Este efecto se analizó con variables derivadas y no con variables discretizadas.

**Implicación para el esquema de rangos del trabajo de grado.** En esta simulación, un modelo lineal que recibe solo rangos con *k* ≥ 5 pierde como máximo 0,003 de AUC y 0,004 de F-measure respecto de las variables derivadas, y con *k* = 2 pierde 0,02 y 0,09. Los modelos basados en árboles pierden más y de forma menos regular. Es decir, para un clasificador lineal, reducir la información a rangos conserva casi todo el poder de ordenamiento si hay al menos 5 rangos por variable.

### 5.3 Limitaciones de la extensión
- **Número de corridas.** Se usaron 3 corridas; las diferencias menores a 0,01 de AUC no se pueden interpretar y no se hicieron pruebas de significancia.
- **Cortes por cuantiles.** Los rangos se definen con cuantiles del conjunto de entrenamiento, que maximizan la información por rango. En un esquema con pruebas de conocimiento cero los rangos los fijaría el diseño del circuito, y podrían ser menos informativos.
- **Datos simulados.** Las variables se generan independientes dado la clase y con efectos monótonos, una estructura que favorece a un modelo lineal sobre rangos. El resultado no se debe extrapolar a datos reales sin repetirlo con ellos.
- **Hiperparámetros.** Se mantuvieron los del artículo (RF con `mtry` = 3, CART con `cp` equivalente a 0,01) y no se reajustaron para el nuevo espacio de entrada, lo que puede penalizar a RF y CART.

## 6. Conclusiones (Paso 8)
1. **Recreación.** Se implementaron el LLRE (K-means con *k* = 14, 11 subgrupos conservados, un Lasso-logístico por conjunto balanceado y pesos según el AUC), RF, LLR y CART, con el protocolo del artículo (partición 80 % / 20 %, variables originales y derivadas, AUC y F-measure), sobre datos simulados que reproducen exactamente lo reportado (tamaño de la muestra, conteo de clases y estadísticos de `age`). Se usaron 3 corridas; el artículo usa 30.
2. **La superioridad del LLRE no se reproduce.** Con variables derivadas el AUC es 0,935 (LLR), 0,926 (RF), 0,918 (LLRE) y 0,895 (CART). El F-measure del LLRE con umbral 0,5 (0,333) es menos de la mitad del de los otros tres clasificadores (≈ 0,65). En el artículo el LLRE es el mejor en ambas métricas.
3. **El F-measure bajo del LLRE se debe al umbral de decisión.** Como se entrena con conjuntos balanceados, el LLRE marca como morosos 8.137 de 30.000 clientes con umbral 0,5 (precisión 0,212, recall 0,894). Con el mejor umbral posible sobre prueba (0,83; cota superior, no utilizable en la práctica) su F1 sube a 0,648, similar al de RF y LLR. Su capacidad de ordenar clientes es comparable; lo que no está calibrado para la prevalencia real es la regla de decisión de 0,5.
4. **La ingeniería de variables importa para el modelo lineal.** El AUC de LLR pasa de 0,734 con variables originales a 0,935 con derivadas, mientras que el de RF no cambia (0,925 y 0,926). Es el mismo patrón del artículo (LLR de 0,490 a 0,857) con menor magnitud.
5. **El nivel de señal condiciona las métricas que dependen del umbral.** El AUC simulado (0,89 a 0,93) es mayor que el del artículo (0,77 a 0,86), y con él cambia el F-measure: por ejemplo, CART obtiene 0,653 frente a 0 en el artículo. De los siete patrones cualitativos contrastados, se reproducen cuatro (todos de AUC) y no se reproducen tres (CART, F-measure del LLRE y AUC del LLRE).
6. **Extensión (variables discretizadas en rangos).** Con un modelo lineal, reducir las variables a *k* ≥ 5 rangos cuesta como máximo 0,003 de AUC y 0,004 de F-measure respecto de las variables derivadas; con *k* = 2 cuesta 0,02 y 0,09. RF y CART pierden más y sin un patrón regular. El LLRE conserva el AUC, pero su F-measure con umbral 0,5 depende de la calibración de sus probabilidades.
7. **Implicaciones para el trabajo de grado (ZKP).** (a) En esta simulación, un modelo de *scoring* lineal que recibe rangos con al menos 5 niveles por variable conserva casi todo su poder de ordenamiento. (b) Los valores extremos y los códigos especiales son un riesgo para el modelo lineal (hipótesis de la sección 4.5, no verificada); tanto las banderas como los rangos lo neutralizan en esta simulación. (c) El AUC y las métricas que dependen del umbral se deben reportar por separado, y el umbral se debe calibrar con la prevalencia. (d) Estos resultados provienen de datos simulados y deben confirmarse con datos reales.

### 6.1 Trabajo futuro
- Aumentar a 30 corridas y aplicar la prueba de Friedman con el post-hoc de Nemenyi, como en el artículo.
- Verificar las hipótesis planteadas: ablación sin los códigos 96/98 (efecto sobre LLR), variación de la intensidad de la señal (comportamiento de CART y de RF en F-measure) y reajuste de `mtry` y `cp` para las variables discretizadas.
- Repetir la extensión con rangos fijados por diseño, como los que atestaría un circuito de conocimiento cero, en lugar de cuantiles.
- Ejecutar el mismo código sobre el conjunto de datos real (*Give Me Some Credit*).